# 07 · Evaluación rolling-origin sobre validación

| | |
|---|---|
| **Fase CRISP-DM** | 5 — Evaluación |
| **Objetivo** | Medir el error de los modelos y de los baselines en validación, por horizonte y por celda, y emitir el veredicto frente a los criterios de éxito. |
| **Entradas** | `data/gold/series_filtradas.parquet`, `data/gold/origenes_validacion.parquet`, `data/gold/escaladores.json`, `models/bi_noa_lstm.pt`, `models/bi_lstm_baseline.pt` |
| **Salidas** | `reports/metricas.csv`, `reports/evaluacion/` (métricas por celda y por horizonte, figuras) |

**Protocolo**
- **Rolling-origin:** orígenes `t0` cada `EVAL_STRIDE` días desde el último día de train. La ventana de entrada es la VMD **causal** calculada en Gold (puede incluir días de train).
- **Inferencia:** las 362 celdas de un origen en un único forward pass → matriz `[362, 180]`; `V̂ = ancla + σ_Δ · ŷ`.
- **Baselines:** persistencia (el ancla) y tendencia lineal ajustada a la ventana de entrada (30 días filtrados).
- **Métricas** en V, contra la señal **real sin filtrar**, solo en objetivos con calidad `ok`/`interpolado`, dentro de validación y con ventana de entrada usable (completa y sin `fuera_servicio`): RMSE, MAE, MAPE y R² (por celda, luego promediado).
- Además, se separan los objetivos que **cruzan un salto de nivel** (hay un salto entre `t0` y `t0 + h`) de los que no.

In [ ]:
# Configuración: localiza la raíz del proyecto (local o Colab) y agrega src/ al path
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/Poc-TP1")
except ImportError:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "voltcast").is_dir())
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message="Mean of empty slice")  # celdas sin objetivos en un subconjunto

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

from voltcast import cleaning as cl
from voltcast import config, io
from voltcast import dataset as ds
from voltcast import metrics as mt
from voltcast import model as mdl

config.ensure_dirs()
EVAL_DIR = config.REPORTS_DIR / "evaluacion"
EVAL_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COLORES = {"Bi-NOA-LSTM": "tab:blue", "Bi-LSTM estándar": "tab:orange", "Persistencia": "k", "Tendencia lineal": "tab:gray"}


def savefig(fig, name):
    path = EVAL_DIR / f"{name}.png"
    fig.savefig(path, dpi=130, bbox_inches="tight")
    print(f"Figura guardada: {path.relative_to(config.BASE_DIR)}")

## 1. Datos de validación y ventanas causales

In [ ]:
gold = io.read_layer("gold", "series_filtradas.parquet")
orig = io.read_layer("gold", "origenes_validacion.parquet")
esc = io.read_json("gold", "escaladores.json")
SIGMA = esc["sigma_delta"]
CELLS = sorted(gold["celda"].astype(str).unique())
L, H, C = config.LOOKBACK_DAYS, config.HORIZON_DAYS, len(CELLS)

V = ds.pivot(gold, "voltaje", CELLS)
FECHAS = V.index
Q = ds.pivot(gold, "calidad", CELLS)
VALIDO = Q.isin(cl.CALIDAD_VALIDA).to_numpy()
OOS = (Q == "fuera_servicio").to_numpy()
KA, VC, VR = (ds.pivot(gold, c, CELLS).to_numpy() for c in ("kA_z", "vcel_z", "voltaje"))
VAL_DESDE = gold.loc[gold["split"] == "val", "fecha"].min()

o = orig.assign(celda=orig["celda"].astype(str)).sort_values(["fecha_origen", "celda", "offset"])
ORIGENES = pd.DatetimeIndex(o["fecha_origen"].unique())
O = len(ORIGENES)
W = o["v_filtrada"].to_numpy().reshape(O, C, L)  # ventanas VMD causales [O, C, L]
assert (o["celda"].to_numpy().reshape(O, C, L)[0, :, 0] == np.array(CELLS)).all()

p0 = FECHAS.get_indexer(ORIGENES)
win = p0[:, None] + np.arange(-L + 1, 1)          # [O, L]
tgt = p0[:, None] + np.arange(1, H + 1)           # [O, H]
dentro = tgt < len(FECHAS)
tgt_c = np.minimum(tgt, len(FECHAS) - 1)
REAL = VR[tgt_c].transpose(0, 2, 1)                # [O, C, H] voltaje real sin filtrar
entrada_ok = np.isfinite(W).all(-1) & ~OOS[win].transpose(0, 2, 1).any(-1)   # [O, C]
MASK = VALIDO[tgt_c].transpose(0, 2, 1) & dentro[:, None, :] & entrada_ok[..., None] & np.isfinite(REAL)
print(f"{O} orígenes ({ORIGENES[0].date()} → {ORIGENES[-1].date()}) × {C} celdas × {H} horizontes")
print(f"Ventanas de entrada usables: {entrada_ok.mean():.1%} | objetivos evaluables: {MASK.sum():,}")
print(f"Orígenes con horizonte completo de 180 días: {(tgt[:, -1] < len(FECHAS)).sum()}")

In [ ]:
# Saltos de nivel (regla R9) sobre la serie real de Gold: solo para separar el análisis del error
saltos = cl.level_shifts(V, Q.apply(lambda s: pd.Categorical(s, categories=cl.CALIDAD).codes))
NXT = ds.next_jump_index(FECHAS, CELLS, saltos)
CRUZA = NXT[p0][:, :, None] <= tgt[:, None, :]    # hay un salto en (t0, t0 + h]
print(f"Saltos en validación: {(saltos['fecha'] >= VAL_DESDE).sum()} | "
      f"objetivos evaluables que cruzan un salto: {(MASK & CRUZA).sum() / MASK.sum():.1%}")

## 2. Inferencia y baselines

In [ ]:
PRED = {}
EL_FLAG = np.array([float(c[0] == "B") for c in CELLS])
for nombre, archivo in [("Bi-NOA-LSTM", "bi_noa_lstm.pt"), ("Bi-LSTM estándar", "bi_lstm_baseline.pt")]:
    m, ck = mdl.load_checkpoint(config.MODELS_DIR / archivo, DEVICE)
    assert ck["cells"] == CELLS and ck["escaladores"]["sigma_delta"] == SIGMA
    out = np.empty((O, C, H))
    for i in range(O):  # un forward pass por origen con las 362 celdas
        out[i], _ = mdl.predict_volts(m, W[i], VALIDO[win[i]].T, KA[win[i]].T, VC[win[i]].T, SIGMA,
                                      np.arange(C), EL_FLAG, DEVICE)
    PRED[nombre] = out
    print(f"{nombre}: entrenado hasta {ck['entrenado_hasta']}, época {ck['mejor_epoca']} → pronósticos {out.shape}")

ANCLA = ds.anchor_from_window(W)
PRED["Persistencia"] = mt.persistence(ANCLA, H)
PRED["Tendencia lineal"] = mt.linear_trend(W, H)

## 3. Tabla comparativa a 60 y 180 días

In [ ]:
SUBCONJ = {"todos": MASK, "sin salto": MASK & ~CRUZA, "cruza salto": MASK & CRUZA}
tablas = []
for sub, mk in SUBCONJ.items():
    for h in config.EVAL_HORIZONS:
        tablas.append(mt.summary_at(PRED, REAL, mk, h).assign(subconjunto=sub))
metricas = pd.concat(tablas, ignore_index=True)
metricas["RMSE_mV"] = metricas["RMSE_V"] * 1000
metricas["MAE_mV"] = metricas["MAE_V"] * 1000
metricas.to_csv(config.REPORTS_DIR / "metricas.csv", index=False)
print(f"✔ {(config.REPORTS_DIR / 'metricas.csv').relative_to(config.BASE_DIR)}")


def vista(sub):
    t = metricas[metricas["subconjunto"] == sub].pivot(index="modelo", columns="horizonte",
                                                        values=["RMSE_mV", "MAE_mV", "MAPE_%", "R2_medio", "R2_mediana"])
    return t.reindex(list(PRED)).round(3)


vista("todos")

In [ ]:
pd.concat({"sin salto": vista("sin salto")[["RMSE_mV", "MAE_mV"]],
           "cruza salto": vista("cruza salto")[["RMSE_mV", "MAE_mV"]]}, axis=1).round(1)

In [ ]:
# ¿Es significativa la diferencia con el mejor baseline? Bootstrap por orígenes (IC 95 %)
filas = []
for h in config.EVAL_HORIZONS:
    base = metricas.query("subconjunto == 'todos' and horizonte == @h and modelo in ['Persistencia', 'Tendencia lineal']")
    mejor_base = base.loc[base["RMSE_V"].idxmin(), "modelo"]
    for nombre in ["Bi-NOA-LSTM", "Bi-LSTM estándar"]:
        d, ci = mt.bootstrap_rmse_diff(PRED[nombre], PRED[mejor_base], REAL, MASK, h)
        filas.append({"horizonte": h, "modelo": nombre, "mejor baseline": mejor_base,
                      "ΔRMSE (mV)": d * 1000, "IC95 inf": ci[0] * 1000, "IC95 sup": ci[1] * 1000,
                      "significativo": ci[1] < 0})
boot = pd.DataFrame(filas)
boot.round(1)

**Lectura.**
- **Todos los objetivos:** a 60 días, Bi-NOA-LSTM (53,0 mV) apenas mejora a la persistencia (54,4 mV), y la diferencia **no es significativa** (IC 95 % de −5,6 a +1,4 mV). A 180 días la mejora sí es **clara y significativa**: 69,3 frente a 77,9 mV, −8,7 mV [−12,4; −5,3]. La tendencia lineal es el peor baseline, porque extrapolar 30 días de pendiente a 60–180 días amplifica el ruido. Por eso el mejor baseline es siempre la persistencia.
- **Sin salto (97 % de los objetivos):** la ventaja del modelo es grande: 23,7 frente a 31,1 mV a 60 días (−24 %) y 42,9 frente a 61,1 mV a 180 días (−30 %). Es el caso para el que se entrenó: la trayectoria de degradación sin intervenciones.
- **Cruza un salto (3 %):** todos los métodos fallan por ~0,2–0,3 V, y la persistencia algo menos que los modelos, porque estos proyectan la subida y el salto es una caída. Ese 3 % de objetivos pesa lo suficiente en el RMSE global como para diluir la ventaja a 60 días.
- **MAPE < 2 %** en todos los casos, porque el voltaje de las celdas ronda los 3 V. El R² por celda es poco informativo: con 40–60 orígenes por celda y un nivel casi constante, el denominador es pequeño y el valor oscila mucho, incluso para la persistencia.
- **Bi-NOA-LSTM frente a Bi-LSTM estándar:** en validación, la NOA es mejor a 60 días (53,0 frente a 57,9 mV) y similar a 180. En el bloque interno de train estaban empatadas, así que con una sola semilla de entrenamiento no se puede atribuir la diferencia a la celda NOA.

## 4. Error vs. horizonte (día 1 → 180)

In [ ]:
curvas = {sub: pd.DataFrame({n: mt.rmse(p, REAL, mk, axis=(0, 1)) * 1000 for n, p in PRED.items()},
                            index=pd.RangeIndex(1, H + 1, name="h")) for sub, mk in SUBCONJ.items()}
pd.concat(curvas, axis=1).to_csv(EVAL_DIR / "rmse_por_horizonte.csv")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
for ax, (sub, cur) in zip(axes, curvas.items()):
    for n in PRED:
        ax.plot(cur.index, cur[n], color=COLORES[n], ls="--" if n in ("Persistencia", "Tendencia lineal") else "-", label=n)
    for h in config.EVAL_HORIZONS:
        ax.axvline(h, color="grey", ls=":", lw=0.8)
    ax.set(xlabel="horizonte (días)", ylabel="RMSE (mV)", title=f"Error vs. horizonte — {sub}")
axes[0].set_ylim(0, curvas["todos"].drop(columns="Tendencia lineal").max().max() * 1.2)
axes[1].set_ylim(0, curvas["sin salto"].drop(columns="Tendencia lineal").max().max() * 1.2)
axes[0].legend(fontsize=8)
fig.tight_layout(); savefig(fig, "01_error_vs_horizonte")
curvas["todos"].iloc[[0, 6, 29, 59, 89, 119, 179]].round(1)

In [ ]:
# Error a 60 y 180 días según la fecha de origen
fig, axes = plt.subplots(1, 2, figsize=(15, 3.8), sharey=False)
for ax, h in zip(axes, config.EVAL_HORIZONS):
    for n in ["Bi-NOA-LSTM", "Persistencia"]:
        r = mt.rmse(PRED[n][..., h - 1], REAL[..., h - 1], MASK[..., h - 1], axis=1) * 1000
        ax.plot(ORIGENES, r, marker=".", color=COLORES[n], label=n)
    frac = (MASK & CRUZA)[..., h - 1].sum(1) / np.maximum(MASK[..., h - 1].sum(1), 1)
    ax2 = ax.twinx(); ax2.bar(ORIGENES, frac * 100, width=5, alpha=0.2, color="crimson"); ax2.set_ylabel("% cruza salto", color="crimson")
    ax.set(ylabel="RMSE (mV)", title=f"h = {h}: RMSE por origen"); ax.legend(loc="upper left", fontsize=8)
fig.tight_layout(); savefig(fig, "02_error_por_origen")

**Lectura.**
- El error crece con el horizonte de forma **suave y sin quiebres**: 12,9 mV a 1 día, 53,0 a 60 y 69,3 a 180. No hay deriva por recursión, porque los 180 días salen de un único pase. El crecimiento refleja la incertidumbre natural del futuro.
- **Sin saltos**, la curva del modelo queda por debajo de la persistencia en todo el rango, y la brecha se amplía con el horizonte. **Con saltos**, el error es de un orden de magnitud mayor y domina el RMSE global.
- **Por origen:** el modelo gana de forma sostenida a 180 días, salvo en los orígenes cuyo horizonte alcanza la intervención de jul-2026 en el EL A (36 celdas). A 60 días, el error de ambos métodos se dispara en los mismos orígenes, los que cruzan saltos.

## 5. Real vs. predicho en celdas críticas

In [ ]:
nivel = pd.Series(np.nanmean(np.where(MASK[..., 0], REAL[..., 0], np.nan), axis=0), index=CELLS)
err180 = pd.Series(mt.rmse(PRED["Bi-NOA-LSTM"][..., 179], REAL[..., 179], MASK[..., 179], axis=0), index=CELLS)
criticas = {"mayor voltaje": nivel.idxmax(), "mayor error a 180 d": err180.idxmax(),
            "segundo mayor voltaje": nivel.drop(nivel.idxmax()).idxmax()}
print({k: f"{c} (V medio {nivel[c]:.3f} V, RMSE180 {err180[c] * 1000:.0f} mV)" for k, c in criticas.items()})

fig, axes = plt.subplots(len(criticas), 1, figsize=(14, 3.4 * len(criticas)), sharex=True)
desde = VAL_DESDE - pd.Timedelta(days=60)
for ax, (tit, c) in zip(axes, criticas.items()):
    j = CELLS.index(c)
    s = V[c][V.index >= desde]
    ax.plot(s.index, s, color="lightgrey", lw=1, label="real")
    for h, mk in [(60, "o"), (180, "s")]:
        ok = MASK[:, j, h - 1]
        f = ORIGENES[ok] + pd.Timedelta(days=h)
        ax.scatter(f, PRED["Bi-NOA-LSTM"][ok, j, h - 1], marker=mk, s=18, color="tab:blue", label=f"Bi-NOA-LSTM h={h}")
        ax.scatter(f, PRED["Persistencia"][ok, j, h - 1], marker=mk, s=18, facecolors="none", edgecolors="k",
                   label=f"persistencia h={h}")
    for d in saltos.loc[(saltos["celda"] == c) & (saltos["fecha"] >= desde), "fecha"]:
        ax.axvline(d, color="crimson", ls=":", lw=1)
    ax.axvline(VAL_DESDE, color="k", ls="--", lw=0.8)
    ax.set(ylabel="V", title=f"{c} — {tit}"); ax.legend(fontsize=7, ncol=5, loc="lower left")
fig.tight_layout(); savefig(fig, "03_real_vs_predicho")

## 6. Error por celda

In [ ]:
por_celda = []
for n, p in PRED.items():
    for h in config.EVAL_HORIZONS:
        k = h - 1
        por_celda.append(pd.DataFrame({
            "celda": CELLS, "modelo": n, "horizonte": h, "n": MASK[..., k].sum(0),
            "RMSE_mV": mt.rmse(p[..., k], REAL[..., k], MASK[..., k], axis=0) * 1000,
            "MAE_mV": mt.mae(p[..., k], REAL[..., k], MASK[..., k], axis=0) * 1000,
            "MAPE_%": mt.mape(p[..., k], REAL[..., k], MASK[..., k], axis=0),
            "R2": mt.r2_per_cell(p[..., k], REAL[..., k], MASK[..., k]),
        }))
por_celda = pd.concat(por_celda, ignore_index=True)
por_celda.to_csv(EVAL_DIR / "metricas_por_celda.csv", index=False)

pc = por_celda.pivot_table(index=["modelo", "horizonte"], columns="celda", values="RMSE_mV")
fig, axes = plt.subplots(2, 1, figsize=(15, 5.5))
for ax, (tit, mat, cmap, centro) in zip(axes, [
        ("RMSE Bi-NOA-LSTM (mV)", pc.loc["Bi-NOA-LSTM"], "viridis", None),
        ("RMSE Bi-NOA-LSTM − persistencia (mV)", pc.loc["Bi-NOA-LSTM"] - pc.loc["Persistencia"], "RdBu_r", 0)]):
    filas = {f"EL {el} · h{h}": mat.loc[h, [c for c in CELLS if c[0] == el]].to_numpy()
             for h in config.EVAL_HORIZONS for el in "AB"}
    sns.heatmap(pd.DataFrame(filas, index=range(1, 182)).T, ax=ax, cmap=cmap, center=centro,
                robust=True, xticklabels=20, cbar_kws={"shrink": 0.8})
    ax.set(xlabel="posición de la celda", title=tit)
fig.tight_layout(); savefig(fig, "04_error_por_celda")

gana = (pc.loc["Bi-NOA-LSTM"] < pc.loc["Persistencia"])
print("Celdas donde Bi-NOA-LSTM supera a la persistencia: " +
      ", ".join(f"h{h}: {gana.loc[h].mean():.0%}" for h in config.EVAL_HORIZONS))

## 7. Veredicto frente a los criterios de éxito

In [ ]:
def rmse_de(modelo, h, sub="todos"):
    return metricas.query("modelo == @modelo and horizonte == @h and subconjunto == @sub")["RMSE_mV"].item()


criterios = []
for h in config.EVAL_HORIZONS:
    mejor_base = min(["Persistencia", "Tendencia lineal"], key=lambda b: rmse_de(b, h))
    b = boot.query("horizonte == @h and modelo == 'Bi-NOA-LSTM'").iloc[0]
    criterios.append({
        "criterio": f"1. Supera al mejor baseline en RMSE a {h} días",
        "resultado": f"{rmse_de('Bi-NOA-LSTM', h):.1f} vs {rmse_de(mejor_base, h):.1f} mV ({mejor_base}); "
                     f"Δ {b['ΔRMSE (mV)']:.1f} mV, IC95 [{b['IC95 inf']:.1f}, {b['IC95 sup']:.1f}]",
        "cumple": rmse_de("Bi-NOA-LSTM", h) < rmse_de(mejor_base, h),
        "significativo": bool(b["significativo"]),
    })
cur = curvas["todos"]["Bi-NOA-LSTM"]
criterios.append({"criterio": "2. Curva de error vs. horizonte reportada, inferencia directa sin recursión",
                  "resultado": f"RMSE {cur.iloc[0]:.1f} mV (h=1) → {cur.iloc[59]:.1f} (h=60) → {cur.iloc[-1]:.1f} (h=180)",
                  "cumple": True, "significativo": None})
criterios.append({"criterio": "3. Métricas en V por celda contra la señal real sin filtrar",
                  "resultado": "reports/metricas.csv y reports/evaluacion/metricas_por_celda.csv", "cumple": True,
                  "significativo": None})
veredicto = pd.DataFrame(criterios)
veredicto

## Veredicto y decisión

| Criterio | Resultado |
|---|---|
| 1a. Supera al mejor baseline a 60 días | **Cumple por poco**: 53,0 frente a 54,4 mV, pero la diferencia no es estadísticamente significativa. |
| 1b. Supera al mejor baseline a 180 días | **Cumple**: 69,3 frente a 77,9 mV (−11 %), significativo. |
| 2. Curva de error vs. horizonte sin deriva | **Cumple**: crecimiento suave de 13 a 69 mV, inferencia directa. |
| 3. Métricas en V contra la señal real | **Cumple** (`reports/metricas.csv`). |

**La PoC cumple formalmente los criterios de éxito**, con una salvedad a 60 días: la ventaja existe, pero es pequeña frente a la variabilidad entre orígenes. El resultado más útil para el negocio es el caso **sin intervención** (97 % de los objetivos). Allí el modelo reduce el error de la persistencia un 24–30 % y sigue la subida por degradación, que es la que determina cuándo una celda superará el voltaje permitido (US007).

**Decisión: continuar, iterando sobre tres puntos antes de una versión productiva:**
1. **Intervenciones:** incorporar el registro de mantenimientos de la planta (reemplazos, puenteos). Sin él, los saltos no se pueden anticipar, y son la principal fuente de error.
2. **Robustez:** entrenar con varias semillas (ensamble) para confirmar si la celda NOA aporta frente al LSTM estándar, que es 5 veces más rápido de entrenar.
3. **Ventanas de entrada tras paros:** algunos pronósticos aislados se desvían cuando la ventana incluye el transitorio de un arranque. Conviene un ancla robusta (mediana) o excluir esos días del ancla.

**Siguiente:** notebook 08 — Despliegue (pronóstico desde la última fecha, `predecir(fecha, celda)` y plan de monitoreo).